### GATConv with Softmax Normalization

Custom GATConv layer ensuring that attention coefficients sum to one for each node, naturally limiting the influence of any single high-degree node.

$$
\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}
$$

where $ e_{ij} $ is the raw attention score between nodes $ i $ and $ j $.


In [ ]:
class SoftNormGATConv(GATConv):
    def forward(self, x, edge_index, return_attention_weights=False):
        # Call the original forward method to compute attention
        out = super().forward(x, edge_index, return_attention_weights=True)
        x, (edge_index, alpha) = out

        # Apply softmax normalization to the attention coefficients
        alpha = F.softmax(alpha, dim=1)

        # If return_attention_weights is True, return x and alpha, else return only x
        if return_attention_weights:
            return x, (edge_index, alpha)
        else:
            return x

### GATConv with Degree-Based Regularization and Normalization of Attention Weights

Incorporate a degree-based regularization term to penalize high-degree nodes in the attention mechanism, ensuring that nodes with high degrees do not disproportionately influence the model's predictions. The degree-regularized attention mechanism can be represented as:

$$
\alpha_{ij} = \frac{\exp\left(\frac{e_{ij}}{\sqrt{d_i}}\right)}{\sum_{k \in \mathcal{N}(i)} \exp\left(\frac{e_{ik}}{\sqrt{d_i}}\right)}
$$

where $ d_i $ is the degree of node $ i $, and $ e_{ij} $ is the raw attention score between nodes $ i $ and $ j $.


In [ ]:
class DegNormGATConv(GATConv):
    def forward(self, x, edge_index, return_attention_weights=False):
        # Call the original forward method to compute attention
        out = super().forward(x, edge_index, return_attention_weights=True)
        x, (edge_index, alpha) = out

        # Get the degrees of the nodes
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float("inf")] = 0

        # Degree penalization
        alpha = alpha / (deg_inv_sqrt[row] * deg_inv_sqrt[col]).unsqueeze(1)

        # Apply softmax normalization to the attention coefficients
        alpha = F.softmax(alpha, dim=1)

        # If return_attention_weights is True, return x and alpha, else return only x
        if return_attention_weights:
            return x, (edge_index, alpha)
        else:
            return x